### Locomotion-Transformer  
### How to  
1. Gazebo を起動  
$ __NV_PRIME_RENDER_OFFLOAD=1 __GLX_VENDOR_LIBRARY_NAME=nvidia ros2 launch mini_pupper_simulation bringup.launch.py launch_twist_converter:=False  


In [ ]:
import os
import sys
# torchをインポートする前に設定
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from gazebo_env import MiniPupperEnv,MAX_ACTION_RAD
from stable_baselines3 import PPO
import rclpy
import torch

import torch as th
import torch.nn as nn
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor


In [ ]:
rclpy.init()
env = MiniPupperEnv()

cont_f=True

if MAX_ACTION_RAD==1.0:
    use_sde=True
    sde_sample_freq=16
    out_dir="outs-10"
else:
    use_sde=True       # add by nishi 2026.8.3
    sde_sample_freq=4   # add by nishi 2026.8.3
    out_dir="outs-05"

CHECKPOINT_DIR = F"{out_dir}/checkpoints/"

if not cont_f:
    reset_num_timesteps=True
else:
    reset_num_timesteps=False

In [ ]:
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
#from lerobot.datasets.lerobot_dataset import LeRobotDataset

# =====================================================================
# 1. 時系列対応版 MiniPupperLocomotionTransformer の定義
# =====================================================================
#---
# new
#---
class MiniPupperLocomotionTransformer(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256, history_len=6):
        super().__init__(observation_space, features_dim)
        
        self.history_len = history_len
        self.num_legs = 4
        self.features_per_leg = 6  
        self.token_dim = 64      
        
        # --- 各入力パーツをトークンに変換する層 ---
        self.cmd_embed = nn.Linear(3, self.token_dim)      
        self.leg_embed = nn.Linear(self.features_per_leg, self.token_dim)
        #self.quat_embed = nn.Linear(4, self.token_dim)     
        # 💡【変更】クォータニオン(4) + 角速度(3) = 7次元 をまとめて64次元のトークンに埋め込む
        self.imu_embed = nn.Linear(7, self.token_dim)     
        
        # --- 【時間軸】位置エンコーディング（形状を [6, 1, 1, 64] にしてブロードキャストを安定化） ---
        self.temporal_embedding = nn.Parameter(th.randn(self.history_len, 1, 1, self.token_dim))
        
        # --- Transformerエンコーダー（改善版パラメータ） ---
        self.leaky_relu_fn = nn.LeakyReLU(negative_slope=0.01, inplace=True)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.token_dim,   # 64
            nhead=2,                  # 1ヘッドあたり32次元を確保してアテンションを安定化
            dim_feedforward=256,      # d_modelの4倍の容量を確保
            batch_first=True,
            activation=self.leaky_relu_fn,
            dropout=0.1               # 状況に応じて 0.0 も検討
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # --- 最終出力をまとめる層 ---
        # トークン数は変わらず 6ステップ × (cmd(1) + leg(4) + imu(1)) = 36個 のまま
        total_tokens = self.history_len * (1 + self.num_legs + 1)  # 6 × 6 = 36
        self.output_layer = nn.Sequential(
            nn.Linear(total_tokens * self.token_dim, features_dim),
            nn.LeakyReLU(negative_slope=0.01, inplace=True)
        )

    def forward(self, observations):
        batch_size = observations.shape[0]
        all_temporal_tokens = []
        
        for t in range(self.history_len):
            # 💡 新しい入力の形状: [batch_size, 34]
            step_obs = observations[:, t, :]  
            
            # --- 【変更】スライス範囲を34次元に合わせて修正 ---
            cmd_vel_data     = step_obs[:, 0:3]    # [0, 1, 2]
            joint_pos        = step_obs[:, 3:15]   # [3 ... 14]
            joint_vel        = step_obs[:, 15:27]  # [15 ... 26]
            quat_data        = step_obs[:, 27:31]  # [27, 28, 29, 30]
            ang_vel_data     = step_obs[:, 31:34]  # 💡【追加】[31, 32, 33] (IMU角速度)
            
            # トークン変換
            cmd_token = self.cmd_embed(cmd_vel_data).unsqueeze(1)
            
            leg_pos_split = joint_pos.view(batch_size, self.num_legs, 3)
            leg_vel_split = joint_vel.view(batch_size, self.num_legs, 3)
            legs_combined = th.cat([leg_pos_split, leg_vel_split], dim=2)
            leg_tokens = self.leg_embed(legs_combined)
            
            # 💡【変更】クォータニオンと角速度を結合して一つのIMUトークンにする [batch_size, 7] ➔ [batch_size, 1, 64]
            imu_combined = th.cat([quat_data, ang_vel_data], dim=1)
            imu_token = self.imu_embed(imu_combined).unsqueeze(1)
            
            # 空間トークン結合 [batch_size, 6, 64] (内訳: cmd(1) + leg(4) + imu(1) = 6)
            spatial_tokens = th.cat([cmd_token, leg_tokens, imu_token], dim=1)
            
            # 時間エンコーディングの加算
            spatial_tokens = spatial_tokens + self.temporal_embedding[t]
            
            all_temporal_tokens.append(spatial_tokens)
            
        tokens = th.cat(all_temporal_tokens, dim=1)  # [batch_size, 36, 64]
        
        transformer_out = self.transformer(tokens)  
        flat_out = transformer_out.view(batch_size, -1)
        return self.output_layer(flat_out)


In [ ]:
import math
from stable_baselines3 import PPO

# ==========================================
# 1. 途中再開に対応したスケジュール関数（改良版）
# ==========================================
def get_resumable_lr_schedule(
    total_project_timesteps: int,  # プロジェクト全体の総ステップ数（例: 200万）
    already_trained_steps: int = 0, # 【重要】これまでに学習済みのステップ数
    warmup_steps: int = 5000,
    peak_lr: float = 2e-4,
    final_lr: float = 1e-6
):
    """
    セーブ＆ロードでの途中再開に対応した Warmup + Cosine スケジュール
    """
    # 今回の learn() で走らせる総ステップ数
    this_run_timesteps = total_project_timesteps - already_trained_steps

    def lr_schedule(progress_remaining: float) -> float:
        # 1. 今回の実行内での経過ステップ数を計算
        steps_in_this_run = this_run_timesteps * (1.0 - progress_remaining)
        
        # 2. 過去のステップ数と合算して「全歴史での通算ステップ数」を出す
        current_step = already_trained_steps + steps_in_this_run
        
        # 3. 通算ステップ数に基づいてLRを判定
        # フェーズ1: Warmup (通算ステップがまだwarmup_steps未満の場合)
        if current_step < warmup_steps:
            alpha = current_step / warmup_steps
            return peak_lr * alpha
            
        # フェーズ2: Cosine Decay (すでにWarmupが終わっている場合)
        else:
            progress = (current_step - warmup_steps) / (total_project_timesteps - warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            
            cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
            lr = final_lr + (peak_lr - final_lr) * cosine_decay
            return lr

    return lr_schedule


In [ ]:
TOTAL_TIMESTEPS = 2_000_000  # 例: 総学習ステップ数 200万
SAVE_FREQ=20_000  # 保存 steps 間隔  何ステップごとに保存するか

#TOTAL_PROJECT_TIMESTEPS = 2_000_000
#ALREADY_TRAINED_STEPS = 500_000  # ★前回のセーブ時のステップ数を指定
ALREADY_TRAINED_STEPS = 20000  # ★前回のセーブ時のステップ数を指定
#ALREADY_TRAINED_STEPS = 240000  # ★前回のセーブ時のステップ数を指定
#ALREADY_TRAINED_STEPS = 380000  # ★前回のセーブ時のステップ数を指定
#ALREADY_TRAINED_STEPS = 660000  # ★前回のセーブ時のステップ数を指定
#ALREADY_TRAINED_STEPS = 720000  # ★前回のセーブ時のステップ数を指定

#REMAINING_STEPS = TOTAL_PROJECT_TIMESTEPS - ALREADY_TRAINED_STEPS
if not cont_f:
    REMAINING_STEPS = TOTAL_TIMESTEPS
else:
    REMAINING_STEPS = TOTAL_TIMESTEPS - ALREADY_TRAINED_STEPS

import os
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

# ==========================================
# 1. 保存先とコールバックの設定
# ==========================================
#CHECKPOINT_DIR = "./logs/checkpoints/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# チェックポイントコールバックの初期化
checkpoint_callback = CheckpointCallback(
    #save_freq=50_000,                  # ★何ステップごとに保存するか（例: 5万ステップごと）
    save_freq=SAVE_FREQ,                 # ★何ステップごとに保存するか（例: 5万ステップごと）
    save_path=CHECKPOINT_DIR,          # 保存先のフォルダ
    name_prefix="ppo_mini_pupper",     # 保存されるファイル名のプレフィックス
    save_replay_buffer=False,          # PPOはOn-policyなのでReplay Bufferの保存は不要（FalseでOK）
    save_vecnormalize=True             # もしVecNormalize（環境の標準化）を使っている場合は状態も一緒に保存
)

if not cont_f:
    # ==========================================
    # 2. PPOの初期化と適用
    # ==========================================
    #TOTAL_TIMESTEPS = 2_000_000  # 例: 総学習ステップ数 200万
    
    # カスタムスケジュール関数の作成
    #custom_lr = get_transformer_lr_schedule(
    #    total_timesteps=TOTAL_TIMESTEPS,
    #    warmup_steps=5000,       # 最初の5000ステップでピークへ上げる
    #    peak_lr=2e-4,            # Transformer RLでよく使われる控えめなピーク値
    #    final_lr=1e-6
    #)

    # 1. 過去の進捗を引き継いだ新しいスケジュール関数を作成
    resume_lr = get_resumable_lr_schedule(
        #total_project_timesteps=TOTAL_PROJECT_TIMESTEPS,
        total_project_timesteps=TOTAL_TIMESTEPS,
        already_trained_steps=0,
        warmup_steps=5000,
        peak_lr=2e-4,
        final_lr=1e-6
    )

else:
    # ==========================================
    # 2. 途中再開（リザーム）の実行手順
    # ==========================================
    # 例：全体で200万ステップ学習させたいプロジェクトで、
    # 50万ステップの時点でセーブしたモデルをロードして、残り150万ステップを再開する場合
    
    #TOTAL_PROJECT_TIMESTEPS = 2_000_000
    #ALREADY_TRAINED_STEPS = 500_000  # ★前回のセーブ時のステップ数を指定
    #REMAINING_STEPS = TOTAL_PROJECT_TIMESTEPS - ALREADY_TRAINED_STEPS
    #REMAINING_STEPS = TOTAL_TIMESTEPS - ALREADY_TRAINED_STEPS
    
    # 1. 過去の進捗を引き継いだ新しいスケジュール関数を作成
    resume_lr = get_resumable_lr_schedule(
        #total_project_timesteps=TOTAL_PROJECT_TIMESTEPS,
        total_project_timesteps=TOTAL_TIMESTEPS,
        already_trained_steps=ALREADY_TRAINED_STEPS,
        warmup_steps=5000,
        peak_lr=2e-4,
        final_lr=1e-6
    )

In [ ]:
# 1. デバイスの定義 (すでに定義済みの場合は不要)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = "cpu"

# ネットワーク構造のカスタマイズ (おすすめ設定)
policy_kwargs_old = dict(
    net_arch=dict(
        pi=[256, 256],  # 政策（Policy）ネットワーク: 256×2層
        vf=[256, 256]   # 価値（Value）ネットワーク: 256×2層
    )
)

# 現在のMlpPolicyの指定をベースに、policy_kwargsを追記します
policy_kwargs_old2 = dict(
    features_extractor_class=MiniPupperLocomotionTransformer,
    features_extractor_kwargs=dict(features_dim=256),
    # 必要に応じて、Transformerの後のActor/Criticそれぞれの全結合層(頭)の厚みを指定
    #net_arch=dict(pi=[128, 128], vf=[128, 128]) 
    net_arch=dict(pi=[256, 256], vf=[256, 256]) 
)

# ポリシー設定にカスタム特徴抽出器を登録
policy_kwargs = dict(
    features_extractor_class=MiniPupperLocomotionTransformer,
    features_extractor_kwargs=dict(features_dim=256, history_len=6),
    # 必要に応じて、Transformerの後のActor/Criticそれぞれの全結合層(頭)の厚みを指定
    #net_arch=dict(pi=[128, 128], vf=[128, 128]) 
    #net_arch=dict(pi=[256, 256], vf=[256, 256]) 
    # 変更案：Actor(pi)は滑らかな制御のために256のまま、状況判断(vf)をより深くする
    net_arch=dict(pi=[256, 256], vf=[512, 512]) 
)

if not cont_f:
    if False:
        model = PPO(
            policy="MlpPolicy",
            env=env,
            policy_kwargs=policy_kwargs,
            #learning_rate=3e-4,
            learning_rate=resume_lr,  # ★ここに作成したスケジュール関数をそのまま渡す
            verbose=1,
            device="cuda",
            use_sde=use_sde,       # add by nishi 2026.8.3
            sde_sample_freq=sde_sample_freq,   # add by nishi 2026.8.3
        )

    if True:
        model = PPO(
            policy="MlpPolicy",           # "MlpPolicy" は、内部で ActorCriticPolicy をコールする。
            env=env,
            policy_kwargs=policy_kwargs,
            #learning_rate=2e-4,        # Transformer向けに少しだけ慎重に設定（2e-4などでも可）
            learning_rate=resume_lr,  # ★ここに作成したスケジュール関数をそのまま渡す
            n_steps=4096,              
            batch_size=128,            
            n_epochs=10,               
            gamma=0.99,                
            gae_lambda=0.95,           
            clip_range=0.2,            
            ent_coef=0.05,             # 積極的な探索設定を維持
            verbose=1,
            device=device,
            #use_sde=True,                # SDEとの相性も抜群です
            #sde_sample_freq=4,   
            use_sde=use_sde,       # add by nishi 2026.8.3
            sde_sample_freq=sde_sample_freq,   # add by nishi 2026.8.3
        )
   
    if False:
        model = PPO(
            ActorCriticPolicy,           # "MlpPolicy" からクラスオブジェクトに変更
            env,
            policy_kwargs=policy_kwargs,
            #learning_rate=2e-4,        # Transformer向けに少しだけ慎重に設定（2e-4などでも可）
            learning_rate=resume_lr,  # ★ここに作成したスケジュール関数をそのまま渡す
            n_steps=4096,              
            batch_size=128,            
            n_epochs=10,               
            gamma=0.99,                
            gae_lambda=0.95,           
            clip_range=0.2,            
            ent_coef=0.05,             # 積極的な探索設定を維持
            verbose=1,
            device=device,
            #use_sde=True,                # SDEとの相性も抜群です
            #sde_sample_freq=4,   
            use_sde=use_sde,       # add by nishi 2026.8.3
            sde_sample_freq=sde_sample_freq,   # add by nishi 2026.8.3
        )
else:
    # 2. モデルをロードする
    # ※注意: custom_objectsに新しいlearning_rate関数を渡すことで、ロードされた古い関数を上書きします
    #model = PPO.load(
    #    "ppo_mini_pupper_500k.zip", 
    #    env=env, 
    #    custom_objects={"learning_rate": resume_lr} # ★ここが最大のポイント
    #)
    
    # policy_kwargs は不要（保存されたファイルから自動で読み込まれます）
    #p=F"ppo_mini_pupper_{ALREADY_TRAINED_STEPS}_steps"
    model = PPO.load(
        os.path.join(CHECKPOINT_DIR, F"ppo_mini_pupper_{ALREADY_TRAINED_STEPS}_steps"),
        env=env,
        device=device,
        custom_objects={"learning_rate": resume_lr} # ★ここが最大のポイント
    )

In [ ]:
# 3. 最初から、学習開始
#   reset_num_timesteps=True
# 及び 残りのステップ数で学習を再開
#   reset_num_timesteps=False にすることで、TensorBoard等の内部ログのステップ数も50万から地続きになります
model.learn(
    total_timesteps=REMAINING_STEPS,    # 今回 train する steps数
    reset_num_timesteps=reset_num_timesteps,
    callback=checkpoint_callback  # ★ここにコールバックを投入！
)


In [ ]:
model.save(os.path.join(out_dir, "ppo_minipupper_test_latest"))